In [ ]:
# ─── CELL 1: Install DGL + dependencies ──────────────────────────────────────
import os, sys
!pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html -q
!pip install dgllife rdkit prefetch_generator pyyaml h5py numba scipy scikit-learn tqdm -q
base = '/usr/local/lib/python3.12/dist-packages'
os.makedirs(f'{base}/torchdata/datapipes/iter', exist_ok=True)
os.makedirs(f'{base}/torchdata/dataloader2', exist_ok=True)
for _f in ['torchdata/__init__.py','torchdata/datapipes/__init__.py',
            'torchdata/dataloader2/__init__.py','torchdata/dataloader2/graph.py']:
    open(f'{base}/{_f}','w').write('')
with open(f'{base}/torchdata/datapipes/iter/__init__.py','w') as _f:
    _f.write('class IterDataPipe:\n def __iter__(self): return iter([])\n'
             'class IterableWrapper(IterDataPipe):\n def __init__(self,i=None,**k): pass\n def __iter__(self): return iter([])\n'
             'class Mapper(IterDataPipe): pass\nclass Filter(IterDataPipe): pass\nclass Batcher(IterDataPipe): pass\n')
gb = '/usr/local/lib/python3.12/dist-packages/dgl/graphbolt/__init__.py'
with open(gb,'r') as _f: _c = _f.read()
_c = _c.replace('if not os.path.exists(path):\n        raise FileNotFoundError(\n            f"Cannot find DGL C++ graphbolt library at {path}"\n        )',
               'if not os.path.exists(path):\n        return')
with open(gb,'w') as _f: _f.write(_c)
import dgl, torch; from rdkit import Chem
dgl.graph(([0,1],[1,2])).to('cuda')
print(f'✅ DGL {dgl.__version__} | CUDA {torch.cuda.is_available()} | rdkit OK')

In [ ]:
# ─── CELL 2: Paths + copy project ─────────────────────────────────────────────
import shutil
QM_HDF5_PATH = '/kaggle/input/datasets/maithreyadonthi/misato-qm/QM.hdf5'
DATA_INPUT   = '/kaggle/input/datasets/maithreyadonthi/mlpla-preprocessed-data/data'
MLPLA_INPUT  = '/kaggle/input/datasets/maithreyadonthi/qm-integrated-folder/ML-PLA-Binding-Affinity'
WORK_DIR = '/kaggle/working/ML-PLA'
SRC_DIR  = f'{WORK_DIR}/src'
CONFIG_PATH  = f'{SRC_DIR}/configs/config.yaml'
TRAIN_PATH   = f'{SRC_DIR}/trainer/train.py'
MODEL_SAVE   = f'{SRC_DIR}/model_save'
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
shutil.copytree(MLPLA_INPUT, WORK_DIR)
DATA_LINK = f'{SRC_DIR}/data'
if os.path.islink(DATA_LINK): os.unlink(DATA_LINK)
os.symlink(DATA_INPUT, DATA_LINK)
os.makedirs(MODEL_SAVE, exist_ok=True)
os.chdir(SRC_DIR)
sys.path.insert(0, SRC_DIR)
print(f'✅ Ready. cwd={os.getcwd()}')

In [ ]:
# ─── CELL 3: Patch config + inject QM loader into train.py ────────────────────
# NOTE: model.py already has LayerNorm in qm_mol_proj (NaN fix baked in)
#       train.py already has NaN-safe batch skipping + metrics filtering
#       qm_features.py already has nan_to_num + clip + mol_weight scaling
#       We only need: config values + QM loader injection + num_process
import yaml, ast

# ── 1. Config ─────────────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)
config['save_dir']         = MODEL_SAVE
config['qm_hdf5_path']     = QM_HDF5_PATH
config['use_qm_mol_feats'] = True
config['num_workers']      = 2
config['batch_size']       = 64
config['epoches']          = int(config.get('epochs', config.get('epoches', 1000)))
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f)
print(f"✅ config.yaml (epoches={config['epoches']}, batch={config['batch_size']})")

# ── 2. train.py — inject QM loader + num_process fix ──────────────────────────
with open(TRAIN_PATH) as f:
    code = f.read()

code = code.replace('num_process = 48', 'num_process = 4')

ANCHOR = 'configs = load_config(args.model_config_path)'
if '_init_qm' not in code:
    INJ  = '\n    # ── QM loader ──────────────────────────────────────────────────\n'
    INJ += '    from dataset.graph_constructor import init_qm_loader as _init_qm\n'
    INJ += '    _qm_path = configs.get("qm_hdf5_path")\n'
    INJ += '    if _qm_path and os.path.exists(_qm_path):\n'
    INJ += '        _init_qm(_qm_path)\n'
    INJ += '        print("[QM] Loaded: " + str(_qm_path), flush=True)\n'
    INJ += '    else:\n'
    INJ += '        print("[QM] QM file not found", flush=True)\n'
    INJ += '    # ──────────────────────────────────────────────────────────────\n'
    code = code.replace(ANCHOR, ANCHOR + INJ, 1)
    print('✅ QM loader injected into train.py')
else:
    print('ℹ️  QM loader already present in train.py')

with open(TRAIN_PATH, 'w') as f:
    f.write(code)

# ── 3. Verification ──────────────────────────────────────────────────────────
try:
    ast.parse(code)
    print('✅ train.py syntax OK')
except SyntaxError as e:
    print(f'❌ train.py syntax error: {e}')

# Verify critical fixes are present in the uploaded code
checks = [
    ('model.py LayerNorm',  f'{SRC_DIR}/models/model.py',       'nn.LayerNorm(QM_MOL_FEAT_DIM)'),
    ('train.py NaN skip',   TRAIN_PATH,                         'continue  # skip'),
    ('train.py n==0 guard', TRAIN_PATH,                         'if n == 0:'),
    ('train.py NaN metrics', TRAIN_PATH,                        'np.isfinite(train_pred_flatten)'),
    ('train.py final NaN',  TRAIN_PATH,                         'np.isfinite(test_pred)'),
    ('qm_features clipping', f'{SRC_DIR}/dataset/qm_features.py', 'np.clip(feats'),
]
all_ok = True
for label, path, needle in checks:
    with open(path) as f:
        if needle in f.read():
            print(f'  ✅ {label}')
        else:
            print(f'  ❌ MISSING: {label}')
            all_ok = False

if all_ok:
    print('\n✅ All NaN fixes verified — ready to train')
else:
    print('\n⚠️  Some fixes missing — training may still produce NaN')

In [ ]:
# ─── CELL 4: TRAIN ───────────────────────────────────────────────────────────
os.chdir(SRC_DIR)
!python -u trainer/train.py --model_config_path {CONFIG_PATH}